# Notebook 2 — Vehicle Intervals

**Aim:** reconstruct each vehicle's test history and convert consecutive tests into annualised mileage intervals with quality flags.

**Reads:** `processed_data/cleaned_mot_tests/` (produced by Notebook 1).

**Produces:**
- `processed_data/vehicle_intervals/` — one row per MOT test, with the previous test's date/mileage attached and the annualised mileage for that interval, where one exists. Saved as one Parquet file per year, the same convention as Notebook 1.
- `processed_data/quality_summary_stage2.json` — interval counts and quality-flag counts, so Notebook 7 can report on them without reloading any data.

**Memory approach:** this notebook processes **one year at a time**, just like Notebook 1. The tricky part is that a vehicle's "previous test" can be in an *earlier* year (a test in January 2023 might need to compare against a test from October 2022), so simply loading each year in isolation would miss those cross-year intervals. Instead, a small carry-over table — just each vehicle's most recent test date and mileage so far — is kept between years and used to fill in the first test of the year for every vehicle. This table has one row per vehicle rather than one row per test, so it stays small even though it spans all six years.

Checks that need a vehicle's full history — decreasing odometer readings, unusually short or long intervals, implausibly high annualised mileage — are calculated here, since this is where that history first becomes available.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# config.py (or the config package) lives alongside this notebook; add it to the import path either way it's run from.
for candidate_dir in [Path.cwd(), Path.cwd() / 'mileage_prediction']:
    if (candidate_dir / 'config.py').exists() or (candidate_dir / 'config' / '__init__.py').exists():
        sys.path.insert(0, str(candidate_dir))
        break

import config

print(f'Processed data folder: {config.PROCESSED_DATA_DIR}')
print(f'Valid interval window: {config.MIN_INTERVAL_DAYS}-{config.MAX_INTERVAL_DAYS} days')
print(f'Implausible annualised mileage threshold: {config.IMPLAUSIBLE_ANNUALISED_MILEAGE:,} miles/year')

Processed data folder: /Users/matthewsalter/Documents/Development/eVED work/data-science/processed_data
Valid interval window: 90-450 days
Implausible annualised mileage threshold: 100,000 miles/year


## Step 1: find each year's cleaned file

Check that Notebook 1's output exists for every year before doing any work.

In [2]:
cleaned_input_dir = config.PROCESSED_DATA_DIR / 'cleaned_mot_tests'
if not cleaned_input_dir.exists():
    raise FileNotFoundError(
        f'Could not find {cleaned_input_dir}. Run 01_load_and_clean_data.ipynb first.'
    )

year_input_paths = {
    year: cleaned_input_dir / f'source_year={year}.parquet' for year in config.YEARS_TO_LOAD
}
missing_years = [str(path) for path in year_input_paths.values() if not path.exists()]
if missing_years:
    raise FileNotFoundError(
        'Could not find these expected files. Run 01_load_and_clean_data.ipynb first:\n'
        + '\n'.join(missing_years)
    )

# Only the columns this notebook (and later ones) actually need are loaded.
COLUMNS_TO_LOAD = [
    'test_id', 'vehicle_id', 'test_date', 'test_mileage', 'test_result',
    'make', 'model', 'fuel_type', 'postcode_area', 'first_use_date', 'source_year',
]
CATEGORY_COLUMNS = ['make', 'model', 'fuel_type', 'postcode_area', 'test_result']

for year, path in year_input_paths.items():
    print(f'{year}: {path.name}')


2020: source_year=2020.parquet
2021: source_year=2021.parquet
2022: source_year=2022.parquet
2023: source_year=2023.parquet
2024: source_year=2024.parquet
2025: source_year=2025.parquet


## Step 2: define the per-year interval calculation

Definitions used from here on:

```text
miles_driven = current_test_mileage - previous_test_mileage
days_between_tests = current_test_date - previous_test_date
annualised_mileage = miles_driven * 365.25 / days_between_tests
```

Before this calculation runs, `PRS` records are removed. A PRS record is a remedial retest at the same station, not a separate period of vehicle use, so allowing it into the sequence could make the next genuine MOT test pair with the PRS retest instead of the previous genuine test.

For one year's tests: sort by `vehicle_id`, `test_date`, `test_id`, then for each vehicle compare every test with the one immediately before it *within that year* using `groupby('vehicle_id').shift()`. A vehicle's first test of the year has no earlier test within the year — for those rows, the previous test's date and mileage are looked up from the carry-over table built from earlier years instead.

A vehicle's very first test ever has nothing to compare against at all (no within-year test, nothing in the carry-over table either). Rather than discard this test, its `previous_test_date`/`previous_test_mileage` are taken to be `first_use_date` and 0 miles, and it's flagged `is_first_test_interval` so it can be treated differently in Step 3 — a new car's first test isn't due for 3 years, so that gap is expected to be driven the whole time, unlike a normal interval where a long gap could mean the vehicle sat SORN.


In [3]:
def compute_intervals_for_year(year_df, carry_over_date, carry_over_mileage):
    """Attach previous-test fields for one year, filling in year-boundary crossings from the carry-over table."""
    year_df = year_df.sort_values(['vehicle_id', 'test_date', 'test_id']).reset_index(drop=True)

    by_vehicle = year_df.groupby('vehicle_id', sort=False)
    year_df['previous_test_date'] = by_vehicle['test_date'].shift(1)
    year_df['previous_test_mileage'] = by_vehicle['test_mileage'].shift(1)

    # A vehicle's first test within this year has no within-year previous test;
    # look its last known test up in the carry-over table from earlier years instead.
    # (an empty carry-over table, on the first year processed, has nothing to look up)
    first_test_of_year = year_df['previous_test_date'].isna()
    if not carry_over_date.empty and first_test_of_year.any():
        carried_vehicle_ids = year_df.loc[first_test_of_year, 'vehicle_id']
        year_df.loc[first_test_of_year, 'previous_test_date'] = carried_vehicle_ids.map(carry_over_date)
        year_df.loc[first_test_of_year, 'previous_test_mileage'] = carried_vehicle_ids.map(carry_over_mileage)

    # A vehicle's very first test has no previous test at all (not even a carried-over one).
    # Treat first_use_date as the "previous test", with 0 miles on the clock, rather than discarding it.
    year_df['is_first_test_interval'] = year_df['previous_test_date'].isna()
    year_df.loc[year_df['is_first_test_interval'], 'previous_test_date'] = year_df.loc[year_df['is_first_test_interval'], 'first_use_date']
    year_df.loc[year_df['is_first_test_interval'], 'previous_test_mileage'] = 0

    year_df['days_between_tests'] = (year_df['test_date'] - year_df['previous_test_date']).dt.days
    year_df['miles_driven'] = year_df['test_mileage'] - year_df['previous_test_mileage']
    year_df['annualised_mileage'] = (
        year_df['miles_driven'] * 365.25 / year_df['days_between_tests']
    ).replace([np.inf, -np.inf], np.nan)
    year_df['vehicle_age_years'] = (year_df['test_date'] - year_df['first_use_date']).dt.days / 365.25

    return year_df

## Step 3: flag rather than discard problematic intervals

An interval is flagged, in this priority order, when:

1. This test's own `test_date` is missing (`missing_test_date`) — this can happen if Step 1's date coercion had to blank out an unparseable value.
2. There is no previous test to compare against at all, not even `first_use_date` (`no_prior_test`) — this should be rare, since Step 2 falls back to `first_use_date` for a vehicle's very first test.
3. The odometer decreased (`decreasing_odometer`).
4. No miles were recorded between the tests (`zero_mileage`) — these are retained for auditability but excluded from valid mileage analysis.
5. The interval is shorter than `MIN_INTERVAL_DAYS` (`interval_too_short`) for MOT-to-MOT pairs. First-use-to-first-test intervals are not subject to this minimum.
6. The interval is longer than `MAX_INTERVAL_DAYS`, or `MAX_INTERVAL_DAYS_FIRST_TEST` for `is_first_test_interval` rows (`interval_too_long`).
7. The resulting annualised mileage is implausibly high (`implausible_mileage`).

Anything left over is `valid` — these are the intervals used as evidence in later notebooks, including first-test intervals now that they're checked against their own, longer window. Nothing is deleted here; every retained row keeps its flag so later notebooks (and this one's summary) can see exactly why an interval was or wasn't used. PRS rows are counted separately and excluded before pairing because they should not participate in the test sequence at all.


In [4]:
def assign_quality_flag(year_df):
    """Add interval_quality_flag to one year's rows. Returns (year_df, flag_counts)."""
    too_short = (~year_df['is_first_test_interval']) & (year_df['days_between_tests'] < config.MIN_INTERVAL_DAYS)
    too_long = np.where(
        year_df['is_first_test_interval'],
        year_df['days_between_tests'] > config.MAX_INTERVAL_DAYS_FIRST_TEST,
        year_df['days_between_tests'] > config.MAX_INTERVAL_DAYS,
    )

    conditions = [
        year_df['test_date'].isna(),
        year_df['previous_test_date'].isna(),
        year_df['miles_driven'] < 0,
        year_df['miles_driven'] == 0,
        too_short,
        too_long,
        year_df['annualised_mileage'] > config.IMPLAUSIBLE_ANNUALISED_MILEAGE,
    ]
    labels = [
        'missing_test_date',
        'no_prior_test',
        'decreasing_odometer',
        'zero_mileage',
        'interval_too_short',
        'interval_too_long',
        'implausible_mileage',
    ]
    year_df['interval_quality_flag'] = np.select(conditions, labels, default='valid')
    year_df['interval_quality_flag'] = year_df['interval_quality_flag'].astype('category')
    return year_df, year_df['interval_quality_flag'].value_counts()


## Step 4: run the year-by-year rolling computation

Process each year in order: load it, remove PRS retests before any pairing, compute intervals (using the carry-over table for anything crossing into an earlier year), flag them, save them, then update the carry-over table with this year's most recent non-PRS test per vehicle before moving to the next year. Quality-flag counts, excluded PRS counts, and each vehicle's running valid-interval count are accumulated as small summaries rather than kept as full per-row data.


In [ ]:
config.ensure_processed_data_dir()
intervals_output_dir = config.PROCESSED_DATA_DIR / 'vehicle_intervals'
intervals_output_dir.mkdir(parents=True, exist_ok=True)

# One row per vehicle: its most recent test date/mileage seen so far, carried between years.
carry_over_date = pd.Series(dtype='datetime64[ns]')
carry_over_mileage = pd.Series(dtype='float32')

flag_counts_total = {}
valid_counts_running = pd.Series(dtype='int64')
invalidated_date_counts = {'test_date': 0, 'first_use_date': 0}
excluded_test_result_counts = {result: 0 for result in config.EXCLUDED_TEST_RESULTS}
valid_first_test_intervals = 0

for year in config.YEARS_TO_LOAD:
    print(f'Processing {year}...')
    year_df = pd.read_parquet(year_input_paths[year], columns=COLUMNS_TO_LOAD)
    for column in CATEGORY_COLUMNS:
        year_df[column] = year_df[column].astype('category')

    # A handful of rows in Notebook 1's output ended up with date columns stored as text
    # rather than real dates; coerce them here so a stray bad value becomes missing
    # rather than crashing the whole year's calculation. Rows a coercion turns into a
    # missing date lose that column's use as evidence, so they're counted here.
    for date_column in ['test_date', 'first_use_date']:
        if not pd.api.types.is_datetime64_any_dtype(year_df[date_column]):
            missing_before = int(year_df[date_column].isna().sum())
            year_df[date_column] = pd.to_datetime(year_df[date_column], errors='coerce')
            missing_after = int(year_df[date_column].isna().sum())
            invalidated_date_counts[date_column] += missing_after - missing_before

    excluded_test_rows = year_df['test_result'].isin(config.EXCLUDED_TEST_RESULTS)
    excluded_this_year = int(excluded_test_rows.sum())
    for result, count in year_df.loc[excluded_test_rows, 'test_result'].value_counts().items():
        excluded_test_result_counts[str(result)] = excluded_test_result_counts.get(str(result), 0) + int(count)
    year_df = year_df.loc[~excluded_test_rows].copy()

    year_df = compute_intervals_for_year(year_df, carry_over_date, carry_over_mileage)
    year_df, flag_counts = assign_quality_flag(year_df)

    year_output_path = intervals_output_dir / f'source_year={year}.parquet'
    year_df.to_parquet(year_output_path, index=False)

    for flag, count in flag_counts.items():
        flag_counts_total[flag] = flag_counts_total.get(flag, 0) + int(count)

    is_valid = year_df['interval_quality_flag'] == 'valid'
    valid_this_year = year_df.loc[is_valid].groupby('vehicle_id', sort=False).size()
    valid_counts_running = valid_counts_running.add(valid_this_year, fill_value=0)

    valid_first_test_intervals += int((is_valid & year_df['is_first_test_interval']).sum())

    # Update the carry-over table with each vehicle's latest non-PRS test this year.
    last_this_year = year_df.groupby('vehicle_id', sort=False).agg(
        last_test_date=('test_date', 'last'), last_test_mileage=('test_mileage', 'last')
    )
    carry_over_date = last_this_year['last_test_date'].combine_first(carry_over_date)
    carry_over_mileage = last_this_year['last_test_mileage'].combine_first(carry_over_mileage)

    print(f'  {len(year_df):,} rows saved. PRS rows excluded before pairing: {excluded_this_year:,}. Valid intervals: {int(flag_counts.get("valid", 0)):,}')
    del year_df

print('\nAll years processed and saved to', intervals_output_dir)
print(f'Rows invalidated by unparseable dates: {invalidated_date_counts}')
print(f'PRS rows excluded before pairing: {sum(excluded_test_result_counts.values()):,}')
print(f'Valid intervals that came from a first-ever test (using first_use_date): {valid_first_test_intervals:,}')


Processing 2020...
  33,447,306 rows saved. PRS rows excluded before pairing: 2,108,569. Valid intervals: 5,530,345
Processing 2021...
  35,941,642 rows saved. PRS rows excluded before pairing: 2,077,671. Valid intervals: 26,218,996
Processing 2022...
  37,179,884 rows saved. PRS rows excluded before pairing: 2,077,375. Valid intervals: 28,327,082
Processing 2023...
  37,863,653 rows saved. PRS rows excluded before pairing: 2,007,031. Valid intervals: 28,965,174
Processing 2024...
  38,626,227 rows saved. PRS rows excluded before pairing: 1,946,425. Valid intervals: 29,374,948
Processing 2025...


## Step 5: save the data-quality summary

With six years of data (2020-2025), a vehicle tested every year can have at most 6 valid intervals — one of which may be its first-ever test, now that those are folded into `annualised_mileage` — used here as a simple proxy for "a complete multi-year history".

In [ ]:
valid_counts_running = valid_counts_running.astype('int64')
max_possible_intervals = len(config.YEARS_TO_LOAD)
vehicles_with_complete_history = int((valid_counts_running == max_possible_intervals).sum())

quality_summary = {
    'usable_vehicle_intervals': int(flag_counts_total.get('valid', 0)),
    'interval_quality_flag_counts': {k: int(v) for k, v in flag_counts_total.items()},
    'excluded_test_result_counts': {k: int(v) for k, v in excluded_test_result_counts.items()},
    'rows_invalidated_by_unparseable_dates': invalidated_date_counts,
    'vehicles_with_at_least_one_valid_interval': int((valid_counts_running > 0).sum()),
    'vehicles_with_complete_multi_year_history': vehicles_with_complete_history,
    'max_possible_valid_intervals': max_possible_intervals,
    'valid_first_test_intervals': valid_first_test_intervals,
}

summary_path = config.PROCESSED_DATA_DIR / 'quality_summary_stage2.json'
with open(summary_path, 'w') as summary_file:
    json.dump(quality_summary, summary_file, indent=2)

print(f'Saved quality summary to {summary_path}')
quality_summary


Saved quality summary to /Users/matthewsalter/Documents/Development/eVED work/data-science/processed_data/quality_summary_stage2.json


{'usable_vehicle_intervals': 157818391,
 'interval_quality_flag_counts': {'interval_too_long': 33966405,
  'valid': 157818391,
  'interval_too_short': 38051599,
  'decreasing_odometer': 4089789,
  'implausible_mileage': 22398,
  'no_prior_test': 215},
 'rows_invalidated_by_unparseable_dates': {'test_date': 0,
  'first_use_date': 1079},
 'vehicles_with_at_least_one_valid_interval': 41389243,
 'vehicles_with_complete_multi_year_history': 3638505,
 'max_possible_valid_intervals': 6,
 'valid_first_test_intervals': 16479968}

## Step 6: sanity-check the saved output

Reload from disk and specifically look for a vehicle whose valid interval **crosses a year boundary** — its `previous_test_date` should fall in an earlier calendar year than its `test_date`. This confirms the carry-over table worked, not just the within-year shift.

In [ ]:
check_df = pd.read_parquet(intervals_output_dir)
print(f'Shape: {check_df.shape}')

crosses_year_boundary = (
    (check_df['interval_quality_flag'] == 'valid')
    & (check_df['previous_test_date'].dt.year < check_df['test_date'].dt.year)
)
example_vehicle_id = check_df.loc[crosses_year_boundary, 'vehicle_id'].iloc[0]

check_df.loc[check_df['vehicle_id'] == example_vehicle_id, [
    'test_date', 'test_mileage', 'previous_test_date', 'previous_test_mileage',
    'days_between_tests', 'annualised_mileage', 'interval_quality_flag',
]]

Shape: (233948797, 18)


,test_date,test_mileage,previous_test_date,previous_test_mileage,days_between_tests,annualised_mileage,interval_quality_flag
7,2020-09-30,3554.0,2017-09-25,0.0,1101.0,1179.017711,valid
35555883,2021-09-24,4040.0,2020-09-30,3554.0,359.0,494.461003,valid
73575194,2022-09-21,5058.0,2021-09-24,4040.0,362.0,1027.139503,valid
112832457,2023-09-28,6731.0,2022-09-21,5058.0,372.0,1642.643145,valid
152703139,2024-09-28,7534.0,2023-09-28,6731.0,366.0,801.354508,valid
193275793,2025-09-17,9286.0,2024-09-28,7534.0,354.0,1807.677966,valid


## Recap and next step

This notebook has produced:

- `processed_data/vehicle_intervals/` — one row per retained MOT test per year, with previous-test fields, annualised mileage, an `interval_quality_flag`, and an `is_first_test_interval` flag marking rows measured from `first_use_date` rather than a previous MOT test. PRS retests are excluded before pairing.
- `processed_data/quality_summary_stage2.json` — valid-interval counts, quality-flag counts including `zero_mileage`, excluded PRS counts, and how many valid intervals came from a first-ever test.

The current interval rules use a 90-day minimum, a 15-month maximum for normal MOT-to-MOT intervals, a longer first-test maximum, a 100,000-mile annualised cap, and an explicit zero-mile exclusion.

Next: **Notebook 3 (`03_mileage_variation.ipynb`)** uses this table on its own to measure how much a vehicle's annual mileage normally varies — independent of the prediction work in Notebooks 4-6.
